# 01 — Data quality check

Before computing any metric, check that the price data in `data/prices.db` is complete and plausible.
Questions:
1. Are there missing values?
2. Are there dates where only some tickers have a price?
3. Are there extreme daily moves, and are they real or data errors?
4. Any other anomalies (weekends, stale prices)?

In [1]:
import pandas as pd
import plotly.express as px

from portfolio.config import load_settings
from portfolio.db import get_connection, load_close_prices

settings = load_settings()
conn = get_connection(settings.database_path)
prices = load_close_prices(conn, settings.all_symbols)
conn.close()

print(prices.shape)
print(prices.index.min().date(), "->", prices.index.max().date())
prices.tail()

(2272, 7)
2018-01-02 -> 2026-11-16


,BCP.LS,EDP.LS,GALP.LS,JMT.LS,NVG.LS,PSI20.LS,SON.LS
date,,,,,,,
2026-11-10,1.1460,4.973,18.745001,19.450001,3.030,9548.459961,2.280
2026-11-11,1.1520,5.058,19.000000,19.490000,3.026,9594.509766,2.265
2026-11-12,1.1460,5.050,19.139999,19.500000,3.036,9596.370117,2.285
2026-11-13,1.1445,5.122,19.355000,19.940001,3.054,9697.639648,2.325
2026-11-16,1.1440,5.088,19.395000,20.020000,3.064,9735.230469,2.335


## 1. Missing values per ticker

In [2]:
prices.isna().sum()

BCP.LS       0
EDP.LS       0
GALP.LS      0
JMT.LS       0
NVG.LS       0
PSI20.LS    12
SON.LS       0
dtype: int64

## 2. Dates where only some tickers have a price

In [3]:
incomplete = prices[prices.isna().any(axis=1)]
print(f"{len(incomplete)} incomplete dates out of {len(prices)} ({len(incomplete) / len(prices):.2%})")
incomplete

12 incomplete dates out of 2272 (0.53%)


,BCP.LS,EDP.LS,GALP.LS,JMT.LS,NVG.LS,PSI20.LS,SON.LS
date,,,,,,,
2018-04-30,0.240162,2.046812,11.108522,11.099640,2.612070,NaN,0.704119
2021-01-04,0.109075,4.070065,6.695621,11.802443,1.661681,NaN,0.484083
2021-01-22,0.104374,4.021576,6.931937,12.049287,1.687060,NaN,0.517771
2021-02-15,0.107508,3.744278,7.462858,10.727212,1.811286,NaN,0.491773
2021-02-17,0.108814,3.752612,7.544782,11.145591,1.801936,NaN,0.508617
2021-03-03,0.104722,3.447282,7.672392,10.835990,1.898110,NaN,0.498730
2021-05-05,0.110293,3.601253,8.054436,12.969367,1.975584,NaN,0.597231
2021-05-11,0.123438,3.452317,7.908708,13.127113,2.105152,NaN,0.609681
2021-05-25,0.136496,3.745461,8.137403,13.583301,2.103446,NaN,0.617390


## 3. Extreme daily moves (|return| > 15%)

In [4]:
returns = prices.pct_change(fill_method=None)
big_moves = returns[(returns.abs() > 0.15).any(axis=1)]
big_moves.round(3)

,BCP.LS,EDP.LS,GALP.LS,JMT.LS,NVG.LS,PSI20.LS,SON.LS
date,,,,,,,
2020-03-09,-0.152,-0.072,-0.165,-0.055,-0.095,-0.087,-0.089
2020-11-09,0.189,0.011,0.174,0.009,0.055,0.045,0.043
2024-04-22,0.051,-0.008,0.206,0.038,0.008,0.035,0.017
2024-07-25,-0.010,-0.000,-0.011,-0.166,-0.025,-0.026,-0.009


## 4. Other sanity checks

In [5]:
print("Weekend dates:", (prices.index.dayofweek >= 5).sum())

# Longest run of identical consecutive closes (long runs = stale / frozen data)
for ticker in prices.columns:
    s = prices[ticker].dropna()
    runs = (s.diff() != 0).cumsum()
    print(f"{ticker:10s} longest run of identical closes: {s.groupby(runs).size().max()}")

Weekend dates: 0
BCP.LS     longest run of identical closes: 3
EDP.LS     longest run of identical closes: 2
GALP.LS    longest run of identical closes: 3
JMT.LS     longest run of identical closes: 3
NVG.LS     longest run of identical closes: 2
PSI20.LS   longest run of identical closes: 1
SON.LS     longest run of identical closes: 3


C:\Users\rodas\AppData\Local\Temp\ipykernel_28348\2934690550.py:1: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  print("Weekend dates:", (prices.index.dayofweek >= 5).sum())


## 5. Visual check — prices rebased to 100

In [6]:
rebased = prices.dropna() / prices.dropna().iloc[0] * 100
px.line(rebased, title="Prices rebased to 100").show()

## Conclusions

- **Missing data:** the 6 stocks have no missing values. The PSI index (PSI20.LS) has no price on 12 of 2,272 dates (0.53%: one in 2018, ten in 2021, one in 2024) while the stocks traded normally, so these are gaps in Yahoo's index feed, not market closures.
- **Extreme moves:** 4 days with a daily move above 15%, all linked to real events:
  - 9 Mar 2020: COVID crash and oil price war (Galp -16.5%, BCP -15.2%)
  - 9 Nov 2020: Pfizer vaccine announcement (BCP +18.9%, Galp +17.4%)
  - 22 Apr 2024: Galp oil discovery in Namibia (+20.6%)
  - 25 Jul 2024: Jeronimo Martins H1 results (-16.6%)

  They are kept: removing them would hide the tail risk the dashboard is meant to measure.
- **Other checks:** no weekend dates; no stale data (at most 3 identical consecutive closes). PSI index volume is 0 on 2 days, which is irrelevant because volume is not used.
- **Decision:** incomplete dates are dropped before computing returns, so every asset and the benchmark are compared on exactly the same days. Cost: 0.53% of observations. Forward-filling the index was rejected because it creates artificial 0% returns followed by a two-day jump, which slightly biases correlation and beta.
